# 2: Taxonomy-Kraken

In [ ]:
# reinstall new version of kraken 
conda create -n kraken2_v2.17.1 kraken2==2.17.1  
# https://github.com/DerrickWood/kraken2/blob/master/docs/MANUAL.markdown
# https://bioconda.github.io/recipes/kraken2/README.html

In [1]:
# Nikea has tried a couple different dbs - see here: https://github.com/nikeaulrich/DR_SCTLD/blob/main/DR_kraken_abundances.ipynb
# Now she is proceeding with PlusPF db
# For now i will use this before trying to build custom db

## Batch Scripts - Kraken

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

# have to wait on PSTR and PAST since I am redoing their assembly steps...
SPP_LIST="MCAV|MMEA|NEG|ORBI"
DBNAME="/datasets/bio/kraken2/PlusPF"
LISTPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
SAMPLELIST="filtered_sample_groups.txt"


# classify each set of paired end reads against Pracken database
while read -r SAMPLEID SPECIE GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    
    # only continue with selected species 
    if [[ ! "$SPECIE" =~ ^($SPP_LIST)$ ]]; then
        echo "Skipping $SAMPLEID ($SPECIE) - not in target list."
        continue
    fi
    echo "Processing ${SPECIE}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly/final_filtered"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    mkdir -p "$OUTDIR"
    
    kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "${SPP_LIST}" processed successfully."

# also do PSTR
SPP="PSTR"
while IFS= read -r SAMPLEID SPECIE GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    
    # only continue with selected species 
    if [[ ! "$SPECIE" = "$SPP" ]]; then
        continue
    fi
    echo "Processing ${SPECIE}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly2/final_filtered"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    
    kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_TASK --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "$SPP" processed successfully."


conda deactivate

# JOB-ID: 
# bash script file name: kraken2-mostspp

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 4:00:00  # Job time limit
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-neg-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

DBNAME="/datasets/bio/kraken2/PlusPF"
SPP="NEG"

FINAL_SAMPLES="7_3_Neg"
READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
# mkdir -p "$OUTDIR"

for SAMPLEID in $FINAL_SAMPLES
do
kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 \
    --report-zero-counts \
    --paired $READS/"${SAMPLEID}_R1.tagged_filter_ready.fastq.gz" \
             $READS/"${SAMPLEID}_R2.tagged_filter_ready.fastq.gz" > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done

echo "Kraken2: All samples in "${SPP}" processed successfully."

In [ ]:
# run past 

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 18:00:00  # Job time limit
#SBATCH --array=1-53%10
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-kraken2-past-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate kraken2_v2.17.1

SPP="PAST"
DBNAME="/datasets/bio/kraken2/PlusPF"
LISTPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}"
SAMPLELIST="spp_samples"

INPUT_FILE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" input.txt)

while read -r SAMPLEID GROUP; do
    # skip header line
    if [[ "$SAMPLEID" == "sampleid" ]]; then
        continue
    fi
    echo "Processing ${SPP}"
    
    READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPECIE}/assembly/final_filtered2"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
    
    kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
    if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples in "$SPP" processed successfully."


conda deactivate

# JOB-ID: 
# bash script file name: kraken2-past

In [ ]:
# try array jobs instead - using PAST
# failed using 1 and 4 hours per sample..trying 8 

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH --cpus-per-task=16
#SBATCH --mem=250G
#SBATCH -t 04:00:00
#SBATCH --array=1-53%10
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/PAST/taxonomy-kraken/slurm-kraken2-past-%A_%a.out

module load conda/latest
conda activate kraken2_v2.17.1

SPP="PAST"

# array job
# sample ID list
SAMPLE_LIST="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/spp_samples"

ARRAY_LIST=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$SAMPLE_LIST")
SAMPLEID=$(echo "$ARRAY_LIST" | cut -f 1)

# paths for kraken
READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered2"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
DBNAME="/datasets/bio/kraken2/PlusPF"

R1="${READS}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz"
R2="${READS}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz"

echo "Processing task ID $SLURM_ARRAY_TASK_ID: Sample $SAMPLEID"
kraken2 --db "$DBNAME" \
        --threads "$SLURM_CPUS_PER_TASK" \
        --report "$OUTDIR/${SAMPLEID}.kreport2" \
        --report-zero-counts \
        --paired "$R1" "$R2" \
        > "$OUTDIR/${SAMPLEID}.kraken2"
if [ $? -eq 0 ]; then
    echo "kraken2 completed successfully for sample: $SAMPLEID"
else
    echo "kraken2 encountered an error for sample: $SAMPLEID"
    exit 1
fi

conda deactivate

# JOB-ID: 61272050 array
# bash script file name: kraken2-past

### Bracken

In [ ]:
# re-install latest udpate
# install in latest kraken2 env (unity has kraken2 as a module but mine is more recently updated)

conda activate kraken2_v2.17.1 
conda install -c bioconda bracken # Bracken v3.01

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH --cpus-per-task=1
#SBATCH --mem=25G
#SBATCH -t 01:00:00
#SBATCH --array=1-222%40
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/taxonomy/kraken/bracken2/slurm-bracken-%A_%a.out

module load conda/latest
conda activate kraken2_v2.17.1 

# Abundance Estimation using bracken from kraken2
# dont need to build bracken db - unity already did it 

# sample input
SAMPLE_LIST="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_samples.txt"
SAMPLEID=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$SAMPLE_LIST")

# parameters
KRAKENFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
LEVEL2="G"
LEVEL3="F"
LEVEL4="O"
READ_LEN=150
DBNAME="/datasets/bio/kraken2/PlusPF"
BRACKEN_PATH2="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL2}"
BRACKEN_PATH3="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL3}"
BRACKEN_PATH4="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL4}"

# run bracken - genus
mkdir -p $BRACKEN_PATH2
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH2/${SAMPLEID}.kreport2_${LEVEL2}.bracken" -r "$READ_LEN" -l "$LEVEL2"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL2} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL2} encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# run bracken - family
mkdir -p $BRACKEN_PATH3
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH3/${SAMPLEID}.kreport2_${LEVEL3}.bracken" -r "$READ_LEN" -l "$LEVEL3"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL3} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL3} encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# run bracken - order
mkdir -p $BRACKEN_PATH4
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH4/${SAMPLEID}.kreport2_${LEVEL4}.bracken" -r "$READ_LEN" -l "$LEVEL4"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL4} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL4} encountered an error for sample: $SAMPLEID"
        exit 1
    fi

conda deactivate 

# JOB-ID: 61280182 species ID (array jobs 1-222), order,fam,genus: 61780631
# bash script file name: bracken.sh

In [ ]:
# had to run the final neg extract alone (in terminal)
 SAMPLEID="Negative_extract_11-2-24"

bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH/${SAMPLEID}.kreport2_${LEVEL1}.bracken" -r "$READ_LEN" -l "$LEVEL1"
if [ $? -eq 0 ]; then
        echo "bracken completed successfully for sample: $SAMPLEID"
    else
        echo "bracken encountered an error for sample: $SAMPLEID"
        exit 1
    fi

    # outputs
 >> Checking for Valid Options...
 >> Running Bracken 
      >> python src/est_abundance.py -i /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/Negative_extract_11-2-24.kreport2 -o /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/Negative_extract_11-2-24.kreport2_S.bracken -k /datasets/bio/kraken2/PlusPF/database150mers.kmer_distrib -l S -t 10
PROGRAM START TIME: 07-01-2026 02:13:44
>> Checking report file: /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/Negative_extract_11-2-24.kreport2
BRACKEN SUMMARY (Kraken report: /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/Negative_extract_11-2-24.kreport2)
    >>> Threshold: 10 
    >>> Number of species in sample: 29924 
          >> Number of species with reads > threshold: 6975 
          >> Number of species with reads < threshold: 22949 
    >>> Total reads in sample: 12863698
          >> Total reads kept at species level (reads > threshold): 9444644
          >> Total reads discarded (species reads < threshold): 8739
          >> Reads distributed: 1302809
          >> Reads not distributed (eg. no species above threshold): 1239
          >> Unclassified reads: 2106267
BRACKEN OUTPUT PRODUCED: /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/Negative_extract_11-2-24.kreport2_S.bracken
PROGRAM END TIME: 07-01-2026 02:13:45
  Bracken complete.
bracken completed successfully for sample: Negative_extract_11-2-24

In [ ]:
## rerun kraken and all bracken levels for single pstr sample ###

In [ ]:
#!/bin/bash
#SBATCH -c 16  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 8:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/taxonomy/kraken/slurm-pstrsample-krakenbracken-%j.out

module load conda/latest
conda activate kraken2_v2.17.1

SPP="PSTR"
SAMPLEID="102019_BEL_CBC_T1_29_PSTR"
DBNAME="/datasets/bio/kraken2/PlusPF"
READS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${SPP}/assembly/final_filtered"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"

echo "Processing ${SPP}: ${SAMPLEID}"

# run kraken
rm -f $OUTDIR/"${SAMPLEID}".kreport2
rm -f $OUTDIR/"${SAMPLEID}".kraken2

kraken2 --db $DBNAME --threads 16 --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
if [ $? -eq 0 ]; then
    echo "kraken2 completed successfully for sample: $SAMPLEID"
else
    echo "kraken2 encountered an error for sample: $SAMPLEID"
    exit 1
fi

# run bracken - all levels S - O 
# parameters
KRAKENFILES="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf"
LEVEL1="S"
LEVEL2="G"
LEVEL3="F"
LEVEL4="O"
READ_LEN=150
DBNAME="/datasets/bio/kraken2/PlusPF"
BRACKEN_PATH1="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken"
BRACKEN_PATH2="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL2}"
BRACKEN_PATH3="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL3}"
BRACKEN_PATH4="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/taxonomy/kraken2_pluspf/bracken/${LEVEL4}"

cd $KRAKENFILES
# run bracken - specie
rm -f "$BRACKEN_PATH1/${SAMPLEID}.kreport2_${LEVEL1}.bracken"
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH1/${SAMPLEID}.kreport2_${LEVEL1}.bracken" -r "$READ_LEN" -l "$LEVEL1"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL1} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL1} encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# run bracken - genus
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH2/${SAMPLEID}.kreport2_${LEVEL2}.bracken" -r "$READ_LEN" -l "$LEVEL2"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL2} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL2} encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# run bracken - family
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH3/${SAMPLEID}.kreport2_${LEVEL3}.bracken" -r "$READ_LEN" -l "$LEVEL3"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL3} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL3} encountered an error for sample: $SAMPLEID"
        exit 1
    fi
# run bracken - order
bracken -d "$DBNAME" -i "$KRAKENFILES/${SAMPLEID}.kreport2" -o "$BRACKEN_PATH4/${SAMPLEID}.kreport2_${LEVEL4}.bracken" -r "$READ_LEN" -l "$LEVEL4"
if [ $? -eq 0 ]; then
        echo "bracken @ level ${LEVEL4} completed successfully for sample: $SAMPLEID"
    else
        echo "bracken @ level ${LEVEL4} encountered an error for sample: $SAMPLEID"
        exit 1
    fi

conda deactivate 

# job id
# job file: kraken-pstr-sample.sh